In [2]:
import pandas as pd

# Week 8 - Step 1: Review dataset and features

# Load the dataset prepared in Week 6/7
# Update this path to match your saved dataset
df = pd.read_csv(
    "../data/processed/rossmann_modeling.csv",
    parse_dates=["Date"]
)
# Features selected for the model
features = [
    "Store",
    "DayOfWeek",
    "Month",
    "Day",
    "Promo",
    "SchoolHoliday",
    "StoreType"
]

target = "Sales"

# Check required columns
required_columns = ["Date"] + features + [target]

print("Missing required columns:")
print([col for col in required_columns if col not in df.columns])

# Dataset information
print("\nDataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

# Check missing values
print("\nMissing values:")
print(df[required_columns].isnull().sum())

# Display feature data types
print("\nFeature data types:")
print(df[features].dtypes)

# Display sample records
display(df[required_columns].head())


Missing required columns:
[]

Dataset shape: (844338, 23)
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00

Missing values:
Date             0
Store            0
DayOfWeek        0
Month            0
Day              0
Promo            0
SchoolHoliday    0
StoreType        0
Sales            0
dtype: int64

Feature data types:
Store            int64
DayOfWeek        int64
Month            int64
Day              int64
Promo            int64
SchoolHoliday    int64
StoreType          str
dtype: object


/var/folders/7d/5cs169vx0mxf0_lp0wfnbrdh0000gn/T/ipykernel_53134/488039473.py:7: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,Date,Store,DayOfWeek,Month,Day,Promo,SchoolHoliday,StoreType,Sales
0,2013-01-01,85,2,1,1,0,1,b,4220
1,2013-01-01,259,2,1,1,0,1,b,6851
2,2013-01-01,262,2,1,1,0,1,b,17267
3,2013-01-01,274,2,1,1,0,1,b,3102
4,2013-01-01,335,2,1,1,0,1,b,2401


In [3]:
# Sort records by date
df = df.sort_values("Date").reset_index(drop=True)

# Split the dataset chronologically
train_data = df[df["Date"] < "2015-05-01"].copy()

validation_data = df[
    (df["Date"] >= "2015-05-01") &
    (df["Date"] < "2015-06-01")
].copy()

test_data = df[df["Date"] >= "2015-06-01"].copy()

# Display results
for name, data in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Testing", test_data)
]:
    print(f"{name} records: {len(data)}")
    print(f"Date range: {data['Date'].min()} to {data['Date'].max()}")
    print()

# Verify that the periods do not overlap
assert train_data["Date"].max() < validation_data["Date"].min()
assert validation_data["Date"].max() < test_data["Date"].min()

assert (
    len(train_data) + len(validation_data) + len(test_data)
    == len(df)
)

print("Chronological split completed successfully!")


Training records: 759848
Date range: 2013-01-01 00:00:00 to 2015-04-30 00:00:00

Validation records: 25879
Date range: 2015-05-01 00:00:00 to 2015-05-31 00:00:00

Testing records: 58611
Date range: 2015-06-01 00:00:00 to 2015-07-31 00:00:00

Chronological split completed successfully!


In [4]:
# Select the same features used in Week 6
features = [
    "Store",
    "DayOfWeek",
    "Month",
    "Day",
    "Promo",
    "SchoolHoliday",
    "StoreType"
]

target = "Sales"

# Separate input features from target
X_train = train_data[features].copy()
y_train = train_data[target]

X_val = validation_data[features].copy()
y_val = validation_data[target]

X_test = test_data[features].copy()
y_test = test_data[target]

# Convert StoreType into numerical columns
X_train = pd.get_dummies(X_train, columns=["StoreType"], dtype=int)
X_val = pd.get_dummies(X_val, columns=["StoreType"], dtype=int)
X_test = pd.get_dummies(X_test, columns=["StoreType"], dtype=int)

# Ensure all datasets use the same columns and order
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Display results
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

print("\nFinal features:")
print(X_train.columns.tolist())

print("\nSample training data:")
display(X_train.head())


Training shape: (759848, 10)
Validation shape: (25879, 10)
Testing shape: (58611, 10)

Final features:
['Store', 'DayOfWeek', 'Month', 'Day', 'Promo', 'SchoolHoliday', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d']

Sample training data:


,Store,DayOfWeek,Month,Day,Promo,SchoolHoliday,StoreType_a,StoreType_b,StoreType_c,StoreType_d
0,85,2,1,1,0,1,0,1,0,0
1,1097,2,1,1,0,1,0,1,0,0
2,769,2,1,1,0,1,0,1,0,0
3,733,2,1,1,0,1,0,1,0,0
4,682,2,1,1,0,1,0,1,0,0


In [5]:
# Columns that should not be used for forecasting
leakage_columns = ["Sales", "Customers"]

# Check whether leakage columns appear in model inputs
for name, X in [
    ("Training", X_train),
    ("Validation", X_val),
    ("Testing", X_test)
]:
    found = [col for col in leakage_columns if col in X.columns]

    print(f"{name} leakage columns: {found}")

    assert len(found) == 0, f"Data leakage detected in {name}!"

# Verify that all datasets have identical features
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

# Check for missing values
print("\nMissing values:")
print("Training:", X_train.isnull().sum().sum())
print("Validation:", X_val.isnull().sum().sum())
print("Testing:", X_test.isnull().sum().sum())

# Verify that no missing values exist
assert not X_train.isnull().any().any()
assert not X_val.isnull().any().any()
assert not X_test.isnull().any().any()

print("\nData leakage and feature checks PASSED!")


Training leakage columns: []
Validation leakage columns: []
Testing leakage columns: []

Missing values:
Training: 0
Validation: 0
Testing: 0

Data leakage and feature checks PASSED!


In [6]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reusable function for evaluating ML models
def evaluate_model(y_actual, y_predicted, model_name):
    mae = mean_absolute_error(y_actual, y_predicted)
    rmse = np.sqrt(mean_squared_error(y_actual, y_predicted))

    print(f"\n{model_name} Evaluation")
    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    }

# Define the evaluation plan
print("BizInsight - Machine Learning Evaluation Plan")
print("--------------------------------------------")
print("1. Train models using the training dataset.")
print("2. Evaluate and compare models on validation data.")
print("3. Select the best model based on validation MAE and RMSE.")
print("4. Evaluate the selected model on the test dataset only after selection.")

print("\nEvaluation function created successfully!")

BizInsight - Machine Learning Evaluation Plan
--------------------------------------------
1. Train models using the training dataset.
2. Evaluate and compare models on validation data.
3. Select the best model based on validation MAE and RMSE.
4. Evaluate the selected model on the test dataset only after selection.

Evaluation function created successfully!


In [7]:
# Check dataset sizes
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

# Check feature consistency
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

# Check that the target is not included in features
assert "Sales" not in X_train.columns
assert "Customers" not in X_train.columns

# Check all features are numerical
assert all(
    np.issubdtype(dtype, np.number)
    for dtype in X_train.dtypes
)

# Check evaluation function using example predictions
sample_actual = np.array([1000, 2000, 3000])
sample_predicted = np.array([1100, 1900, 3200])

sample_result = evaluate_model(
    sample_actual,
    sample_predicted,
    "Workflow Test"
)

assert sample_result["MAE"] >= 0
assert sample_result["RMSE"] >= 0

print("\nBizInsight - Week 8 Final Verification")
print("------------------------------------")
print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))
print("Number of features:", X_train.shape[1])
print("Evaluation function: Working")
print("Data preparation: Complete")
print("Chronological split: Complete")
print("Workflow status: READY FOR WEEK 9")



Workflow Test Evaluation
MAE: 133.33
RMSE: 141.42

BizInsight - Week 8 Final Verification
------------------------------------
Training records: 759848
Validation records: 25879
Testing records: 58611
Number of features: 10
Evaluation function: Working
Data preparation: Complete
Chronological split: Complete
Workflow status: READY FOR WEEK 9
